In [1]:
import csv
import glob
import os
from pprint import pprint

In [2]:
def load_convergence_data(base_dir):
    """load rows grouped by folder."""
    rows_by_folder = {}
    for folder in [f"s{i}" for i in range(10)]:
        rows = []
        for file in glob.glob(os.path.join(base_dir, folder, "version_vector_states.csv")):
            with open(file, newline="") as f:
                reader = csv.reader(f)
                try:
                    _ = next(reader)
                except StopIteration:
                    continue
                for reader_row in reader:
                    ts = int(reader_row[0])
                    values = list(map(int, reader_row[1:]))
                    rows.append((ts, values))
        rows_by_folder[folder] = rows
    return rows_by_folder

def first_occurrence_thresholds(rows, k):
    """return list of (threshold, timestamp) for first occurrence of each value in column k."""
    seen = set()
    result = []
    for ts, values in rows:
        val = values[k]
        if val == 0:
            continue
        if val not in seen:
            seen.add(val)
            result.append((val, ts))
    return result


def process_folder_first_occurrence(data, k):
    """process folder s{k} using first occurrence per threshold in column k."""
    results = []

    thresholds = first_occurrence_thresholds(data[f"s{k}"], k)

    for val, base_ts in thresholds:
        max_ts = 0
        for folder, rows in data.items():
            if folder == f"s{k}":
                continue
            candidate_ts = [ts2 for ts2, vals2 in rows if vals2[k] >= val]
            min_ts = min(candidate_ts) if candidate_ts else None
            if min_ts == None: # This did not converge, so we exclude this
                max_ts = None
                break
            if min_ts > max_ts:
                max_ts = min_ts
        if max_ts != None:
            results.append({
                "from_folder": f"s{k}",
                "base_timestamp": base_ts,
                "threshold": val,
                "max_timestamp_other_folders": max_ts,
                "time_difference(us)": max_ts - base_ts
            })

    return results

def get_convergence_times(base_dir):
    data = load_convergence_data(base_dir)
    results = []

    for i in range(0, 10):
        results += process_folder_first_occurrence(data, i)
            
    return results


In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


networks = ["ring", "full", "super_nodes", "sparse"]
strategies = ["push_one", "pull_one", "pull_log", "pull_full", 
              "pushpull_one", "pushpull_log", "pushpull_full"]

color_map = {
    "push_one": "#1f77b4",      # blue
    "pull_one": "#2ca02c",      # green
    "pull_log": "#66c2a5",      # lighter green
    "pull_full": "#006d2c",     # dark green
    "pushpull_one": "#9467bd",  # purple
    "pushpull_log": "#c2a5cf",  # light purple
    "pushpull_full": "#54278f"  # deep purple
}

legend_names = {
    "push_one": "Proactive",
    "pull_one": "Reactive",
    "pull_log": "Reactive(log)",
    "pull_full": "Reactive(all)",
    "pushpull_one": "Hybrid",
    "pushpull_log": "Hybrid(log)",
    "pushpull_full": "Hybrid(all)",
}

network_names = {
    "ring": "Ring Topology",
    "full": "Fully-Connected Topology",
    "super_nodes": "Super-Nodes Topology",
    "sparse": "Mesh-Like Topology",
}

fig = make_subplots(
    rows=2, cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.01,
    vertical_spacing=0.01
)

subplot_pos = {
    "ring": (1, 1),
    "full": (1, 2),
    "super_nodes": (2, 1),
    "sparse": (2, 2)
}

for network in networks:
    row, col = subplot_pos[network]

    for strategy in strategies:
        if network in ["ring", "super_nodes"] and strategy in ["pull_log", "pushpull_log"]:
            continue

        all_runs_res = []
        for run in range(1, 6):
            all_runs_res += get_convergence_times(
                f"../benchmarks/exp_network_{network}_100_{strategy}_run{run}"
            )
        convergence_times_mikros = [v["time_difference(us)"] / 1000 for v in all_runs_res]

        fig.add_trace(
            go.Box(
                y=convergence_times_mikros,
                name=legend_names[strategy],
                marker=dict(size=4, opacity=0.5, color=color_map[strategy]),
                showlegend=(network == "full"),  # legend only in top-right subplot
                line=dict(width=1.5),
            ),
            row=row, col=col
        )

    # Remove x-axis labels
    fig.update_xaxes(showticklabels=False, row=row, col=col)

    if col == 1:
        fig.update_yaxes(title_text="Convergence Time (ms)", row=row, col=col, title_font=dict(size=12))

    # Plot heading
    fig.add_annotation(
        text=network_names[network],
        xref=f"x{2 if (row,col) != (1,1) else ''} domain",
        yref=f"y{2 if (row,col) != (1,1) else ''} domain",
        x=0.5, y=0.98,  # centered near top
        showarrow=False,
        font=dict(size=14, color="black"),
        row=row, col=col
    ) 


for axis in fig.layout:
    if axis.startswith('yaxis') and axis != 'yaxis':
        fig.layout[axis].update(matches='y')

# Layout tweaks
fig.update_layout(
    height=400, width=800,
    legend=dict(
        font=dict(size=12),
        orientation="h",
        y=-0.01,
        x=0.5,
        xanchor="center",
    ),
    margin=dict(l=0, r=0, t=0, b=45),
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
)
fig.update_xaxes(showline=True, linewidth=1, linecolor="black", ticks=None, tickfont=dict(size=12))
fig.update_yaxes(showline=True, linewidth=1, linecolor="black", ticks="outside", gridcolor="lightgray", gridwidth=0.5, tickfont=dict(size=12))

fig.write_image("./covergence_times.pdf", "pdf", width=800, height=400)
fig.show()
